In [3]:
# imports
from dash import Dash, html, dcc, dash_table, Input, Output, State
import sqlite3
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from textblob import TextBlob
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go
import nltk

ModuleNotFoundError: No module named 'dash'

In [ ]:
# setup
pd.set_option("display.max_colwidth", None) # display full content of cell
nltk.download("stopwords") # Downloads NLTK's stopwords dataset (common words like "the", "is", "and" that are typically filtered out before text analysis).

DB_PATH = r"C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Arkansas Tech Social Media/social_media.db" # path to database

In [ ]:
# load data
conn = sqlite3.connect(DB_PATH) # connect to file path
df = pd.read_sql_query("SELECT * FROM post", conn) # sql query to read table (must be a broad query)
conn.close() # must close connection

In [ ]:
# numeric columns
numeric_cols = ["Likes", "Views", "Shares", "Follows"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
# Ensure datetime column
df['Publish time'] = pd.to_datetime(df['Publish time'], errors='coerce') # make 'Publish time' into date time format
df['date'] = df['Publish time'].dt.date # make the date column

In [ ]:
df = df.dropna(subset=numeric_cols + ["Description"]) #  Removes any rows where there's a missing value in any of the numeric columns or the "Description" column.
df['Post type'] = df['Post type'].fillna("Unknown") # fill with the word "Unknown"
post_types = df['Post type'].unique().tolist() # stores unique value as python text

In [ ]:
# text features
df["Description"] = df["Description"].astype(str) # description as a string
df["sentiment"] = df["Description"].apply(lambda x: TextBlob(x).sentiment.polarity) # Uses TextBlob to score how positive or negative each description is, on a scale from -1 (very negative) to +1 (very positive). 0 is neutral.
df["subjectivity"] = df["Description"].apply(lambda x: TextBlob(x).sentiment.subjectivity) # scores how opinion-based vs. fact-based the text is, from 0 (very objective) to 1 (very subjective).
df["len_chars"] = df["Description"].apply(len) # length of characters column
df["len_words"] = df["Description"].apply(lambda x: len(x.split())) # length of words column

In [ ]:
# cluster
kmeans = KMeans(n_clusters=4, random_state=42) # 4 clusters random state means it can reproduce every run
df["cluster"] = kmeans.fit_predict(df[numeric_cols]) # create a cluster column for every row

In [ ]:
# posting gap columns
df = df.sort_values('Publish time') # sort by publish time
df['days_between_posts'] = df['Publish time'].diff().dt.days.fillna(0) # create a column to be difference between first post
df['total_posts_per_row'] = 1
df['view_like_ratio'] = df.apply(lambda row: row['Views']/row['Likes'] if row['Likes'] > 0 else 0, axis=1) # Divides Views by Likes for each post to measure how many views it takes to get a like.

In [ ]:
post_type_options = [{"label": pt, "value": pt} for pt in post_types] or [{"label": "None", "value": "None"}] # Creates a list of label/value pairs from the post_types list collected earlier. The or [{"label": "None", "value": "None"}] is a fallback in case post_types is empty, ensuring the dropdown always has at least one option.
cluster_options = [{"label": str(c), "value": c} for c in sorted(df["cluster"].unique())] # Does the same for the cluster labels (0, 1, 2, 3). sorted() ensures they appear in order, and str(c) makes the label human-readable while keeping the actual value as a number.
days_options = [{"label": d, "value": d} for d in ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]] # Hardcodes the days of the week as dropdown options, likely for filtering posts by what day they were published.
numeric_cols_options = [{"label": c, "value": c} for c in numeric_cols] # Turns the numeric column names into dropdown options, probably so the user can dynamically pick which metric to plot or analyze.

In [ ]:
# initialize the Dash app using the current module name
# suppress_callback_exceptions prevents errors from callbacks referencing components in inactive tabs
app = Dash(__name__, suppress_callback_exceptions=True)

# define the overall layout of the app wrapped in a single outer div
app.layout = html.Div([
    html.H1("Social Media Dashboard", style={"textAlign": "center"}), # main title centered at the top
    dcc.Tabs([ # container for all tabs

        # Tab 1: Average Follows
        dcc.Tab(label="Average Follows", children=[
            html.Div([
                html.Label("Select Post Type:"), # label for the radio buttons
                dcc.RadioItems(id="post-type-buttons", options=post_type_options, value=post_types[0], inline=True), # radio buttons to filter by post type, defaults to first post type
                html.H2(id="average-follows-display", style={"textAlign": "center", "color": "blue"}), # displays the calculated average follows
                html.Div(id="corrupted-values-display", style={"marginTop": "20px"}) # displays any corrupted/outlier values
            ], style={"margin": "20px"})
        ]),

        # Tab 2: Regression & Clusters
        dcc.Tab(label="Regression & Clusters", children=[
            html.Div([
                html.Label("Select X-axis:"), # label for x-axis dropdown
                dcc.Dropdown(id="x-axis-dropdown", options=numeric_cols_options, value="Likes"), # dropdown to pick x-axis metric, defaults to Likes
                html.Label("Select Y-axis:"), # label for y-axis dropdown
                dcc.Dropdown(id="y-axis-dropdown", options=numeric_cols_options, value="Views"), # dropdown to pick y-axis metric, defaults to Views
                html.Label("Select Graph Type:"), # label for graph type selection
                dcc.RadioItems(id="graph-type-buttons",
                               options=[{"label": "Regression", "value": "regression"},
                                        {"label": "Cluster", "value": "cluster"}],
                               value="regression", inline=True), # toggle between regression and cluster plot, defaults to regression
                html.Br(), # spacing
                html.Label("Filter by Post Type:"), # label for post type filter
                dcc.RadioItems(id="post-type-filter",
                               options=[{"label": "All", "value": "All"}] + post_type_options,
                               value="All", inline=True), # filter data by post type, defaults to All
                html.Label("Gain a Follower:"), # label for follower gain filter
                dcc.RadioItems(id="gain-follower-filter",
                               options=[{"label": "All", "value": "All"}, {"label": "Yes", "value": "Gain"}],
                               value="All", inline=True), # filter to only show posts that gained a follower
                html.Label("Select Cluster(s):"), # label for cluster filter
                dcc.Dropdown(id="cluster-filter", options=cluster_options, multi=True, placeholder="All clusters"), # multi-select dropdown to filter by cluster(s)
                html.Br(), # spacing
                dcc.Graph(id="dynamic-graph"), # placeholder for the regression or cluster graph
                html.H3("Data Table"), # section header for the data table
                dash_table.DataTable(id="dynamic-table",
                                     columns=[{"name": i, "id": i} for i in df.columns], # one column for every column in the dataframe
                                     page_action="none", # load all rows at once (no pagination)
                                     filter_action="native", # allow filtering by column values
                                     sort_action="native", # allow sorting by column
                                     style_table={"overflowX": "auto", "maxHeight": "600px", "overflowY": "scroll"}) # scrollable table
            ], style={"margin": "20px"})
        ]),

        # Tab 3: Random Forest
        dcc.Tab(label="Random Forest", children=[
            html.Div([
                html.Label("Select Post Type:"), # label for post type filter
                dcc.Dropdown(id="rf-posttype-dropdown",
                             options=[{"label": "All", "value": "All"}] + post_type_options,
                             value="All"), # filter data by post type before running the model
                html.Br(), # spacing
                html.Label("Select Target Column:"), # label for target metric dropdown
                dcc.Dropdown(id="rf-target-dropdown", options=numeric_cols_options, value="Likes"), # pick which metric the random forest will try to predict
                html.Br(), # spacing
                dcc.Graph(id="rf-importance-graph") # placeholder for the feature importance bar chart
            ], style={"margin": "20px"})
        ]),

        # Tab 4: Text Analysis
        dcc.Tab(label="Text Analysis", children=[
            html.H3("Text Features"), # section header
            html.Div([
                html.Label("Filter by Minimum Views:"), # label for views filter input
                dcc.Input(id="views-filter-input", type="number", placeholder="Enter min views", style={"marginRight": "20px"}), # number input to filter rows below a minimum view count
                html.Label("Filter by Minimum Likes:"), # label for likes filter input
                dcc.Input(id="likes-filter-input", type="number", placeholder="Enter min likes") # number input to filter rows below a minimum like count
            ], style={"marginBottom": "10px"}),
            dash_table.DataTable(id="text-analysis-table",
                                 columns=[{"name": i, "id": i} for i in ["Description","Permalink"] + numeric_cols + ["sentiment","subjectivity","len_words","len_chars"]], # show description, link, numeric metrics, and text features
                                 data=df.to_dict("records"), # load the full dataframe as the initial data
                                 page_action="none", # no pagination
                                 filter_action="native", # allow column filtering
                                 sort_action="native", # allow column sorting
                                 style_table={"overflowX": "auto", "maxHeight": "600px", "overflowY": "scroll"}), # scrollable table
            html.Button("Download Filtered CSV", id="download-text-csv-btn", style={"marginTop": "15px"}), # button to trigger CSV download
            dcc.Download(id="download-text-csv") # hidden component that handles the file download
        ]),

        # Tab 5: Time Series
        dcc.Tab(label="Time Series", children=[
            html.Div([
                html.Label("Select Metrics:"), # label for metric checklist
                dcc.Checklist(id="ts-metrics-checklist", options=numeric_cols_options, value=["Likes"], inline=True), # checkboxes to select one or more metrics to plot over time
                html.Br(), # spacing
                html.Label("Filter by Post Type:"), # label for post type filter
                dcc.RadioItems(id="ts-post-type-filter", options=[{"label": "All", "value": "All"}] + post_type_options,
                               value="All", inline=True), # filter time series by post type
                html.Label("Gain a Follower:"), # label for follower gain filter
                dcc.RadioItems(id="ts-gain-follower-filter",
                               options=[{"label": "All", "value": "All"}, {"label": "Yes", "value": "Gain"}],
                               value="All", inline=True), # filter to only show posts that gained a follower
                html.Br(), # spacing
                dcc.Graph(id="time-series-graph") # placeholder for the time series graph
            ], style={"margin": "20px"})
        ]),

        # Tab 6: Best Time to Post Heatmap
        dcc.Tab(label="Best Time to Post", children=[
            html.Div([
                html.H3("Best Time to Post by Engagement Metric"), # section header
                html.Label("Select Metric:"), # label for metric dropdown
                dcc.Dropdown(id="heatmap_metric", options=numeric_cols_options, value="Likes"), # pick which metric to aggregate in the heatmap
                html.Br(), # spacing
                html.Label("Filter by Post Type:"), # label for post type filter
                dcc.Dropdown(id="heatmap_posttype", options=[{"label": "All", "value": "All"}] + post_type_options, value="All"), # filter heatmap by post type
                html.Br(), # spacing
                html.Button("Select All Days", id="heatmap_select_all_days", style={"marginBottom": "10px"}), # button to check all days at once
                dcc.Checklist(id="heatmap_day_filter", options=days_options, value=[], inline=True), # checkboxes to filter by day of week
                html.Br(), # spacing
                dcc.Graph(id="heatmap_graph"), # placeholder for the heatmap
            ], style={"margin": "20px"})
        ]),

        # Tab 7: Posting Gap Analysis
        dcc.Tab(label="Posting Gap Analysis", children=[
            html.Div([
                html.H3("Days Between Posts Analysis"), # section header
                html.Label("Filter by Post Type:"), # label for post type filter
                dcc.Dropdown(id="posting-gap-posttype-filter", options=[{"label": "All", "value": "All"}] + post_type_options,
                             value="All"), # filter analysis by post type
                html.Br(), # spacing
                html.Label("Select Metric to Visualize:"), # label for metric radio buttons
                dcc.RadioItems(id="posting-gap-metric",
                               options=[
                                   {"label": "Views", "value": "Views"},
                                   {"label": "Likes", "value": "Likes"},
                                   {"label": "View/Like Ratio", "value": "view_like_ratio"},
                                   {"label": "Total Posts", "value": "total_posts_per_row"}
                               ],
                               value="Views", inline=True), # pick which metric to plot against posting gap
                dcc.Graph(id="posting-gap-graph"), # placeholder for the posting gap vs metric graph
                html.H3("Number of Posts by Days After Previous Post"), # section header for the count graph
                dcc.Graph(id="days-between-post-count-graph") # placeholder for the post count by gap graph
            ], style={"margin": "20px"})
        ])

    ])
])

In [ ]:
# Average Follows callback - triggers when the post type radio button changes
@app.callback(
    Output("average-follows-display", "children"), # updates the average follows text
    Output("corrupted-values-display", "children"), # updates the corrupted values table
    Input("post-type-buttons", "value") # listens for post type selection
)
def update_average(post_type):
    filtered_df = df[df["Post type"] == post_type] # filter dataframe to selected post type
    avg_follows = filtered_df["Follows"].mean() # calculate the average follows for that post type
    avg_text = f"Average Follows for {post_type}: {avg_follows:.2f}" # format the result as a string rounded to 2 decimals
    bad_rows = filtered_df[filtered_df["Follows"].isna()] # find rows where Follows is missing/null
    corrupted_text = html.Div([ # if there are bad rows, build a table to display them
        html.H4("Corrupted Follows values found:"),
        dash_table.DataTable(data=bad_rows.to_dict("records"), page_action="none",
                             style_table={"overflowX": "auto", "maxHeight": "400px", "overflowY": "scroll"})
    ]) if not bad_rows.empty else html.Div() # if no bad rows, return an empty div
    return avg_text, corrupted_text

In [ ]:
# Dynamic Regression & Clusters callback - triggers when any filter or axis selection changes
@app.callback(
    Output("dynamic-graph", "figure"), # updates the scatter/regression/cluster graph
    Output("dynamic-table", "data"), # updates the data table below the graph
    Input("graph-type-buttons", "value"), # listens for regression vs cluster selection
    Input("x-axis-dropdown", "value"), # listens for x-axis metric selection
    Input("y-axis-dropdown", "value"), # listens for y-axis metric selection
    Input("cluster-filter", "value"), # listens for cluster filter selection
    Input("post-type-filter", "value"), # listens for post type filter selection
    Input("gain-follower-filter", "value") # listens for follower gain filter selection
)
def update_dynamic_graph(graph_type, x_col, y_col, selected_clusters, post_type, gain_filter):
    filtered_df = df.copy() # start with a full copy of the dataframe
    if selected_clusters: # if specific clusters are selected, filter to only those
        filtered_df = filtered_df[filtered_df["cluster"].isin(selected_clusters)]
    if post_type != "All": # if a specific post type is selected, filter to only that type
        filtered_df = filtered_df[filtered_df["Post type"] == post_type]
    if gain_filter == "Gain": # if gain filter is active, only keep posts that gained at least 1 follower
        filtered_df = filtered_df[filtered_df["Follows"] >= 1]

    if graph_type == "regression":
        pred_frame = sm.OLS(filtered_df[y_col], sm.add_constant(filtered_df[x_col])).fit().get_prediction(sm.add_constant(filtered_df[x_col])).summary_frame(alpha=0.05) # fit OLS regression and get predictions with 95% intervals
        fig = px.scatter(filtered_df, x=x_col, y=y_col, hover_data=["Description","Permalink"], title=f"Regression: {y_col} ~ {x_col}") # base scatter plot
        fig.add_scatter(x=filtered_df[x_col], y=pred_frame["mean"], mode="lines", name="OLS Fit", line=dict(color="blue")) # add the regression fit line
        fig.add_scatter(x=filtered_df[x_col], y=pred_frame["obs_ci_lower"], mode="lines", name="95% PI Lower", line=dict(color="red", dash="dash")) # add lower prediction interval
        fig.add_scatter(x=filtered_df[x_col], y=pred_frame["obs_ci_upper"], mode="lines", name="95% PI Upper", line=dict(color="red", dash="dash")) # add upper prediction interval
    else:
        fig = px.scatter(filtered_df, x=x_col, y=y_col, color="cluster", # color points by cluster label
                         hover_data=["Description","Permalink","sentiment","subjectivity","len_words","len_chars"], # show text features on hover
                         title=f"Cluster: {y_col} vs {x_col}")

    table_data = filtered_df.to_dict("records") # convert filtered dataframe to dict format for the data table
    return fig, table_data

In [ ]:
# Random Forest Importance callback - triggers when target metric or post type changes
@app.callback(
    Output("rf-importance-graph", "figure"), # updates the feature importance bar chart
    Input("rf-target-dropdown", "value"), # listens for target metric selection
    Input("rf-posttype-dropdown", "value") # listens for post type filter selection
)
def update_rf_importance(target_col, post_type):
    filtered_df = df.copy() # start with a full copy of the dataframe
    if post_type != "All": # if a specific post type is selected, filter to only that type
        filtered_df = filtered_df[filtered_df["Post type"] == post_type]
    features = [c for c in numeric_cols if c != target_col] # use all numeric columns except the target as features
    rf = RandomForestRegressor(n_estimators=100, random_state=42) # initialize random forest with 100 trees
    rf.fit(filtered_df[features], filtered_df[target_col]) # train the model to predict the target column
    importance_df = pd.DataFrame({"Feature": features, "Importance": rf.feature_importances_}).sort_values("Importance", ascending=False) # build a sorted dataframe of feature importances
    return px.bar(importance_df, x="Feature", y="Importance", title=f"Random Forest Feature Importance for {target_col} ({'All' if post_type=='All' else post_type})") # return bar chart of importances



In [ ]:
# Time Series callback - triggers when metrics, post type, or follower gain filter changes
@app.callback(
    Output("time-series-graph", "figure"), # updates the time series graph
    Input("ts-metrics-checklist", "value"), # listens for selected metrics
    Input("ts-post-type-filter", "value"), # listens for post type filter
    Input("ts-gain-follower-filter", "value") # listens for follower gain filter
)
def update_time_series(metrics, post_type, gain_filter):
    filtered_df = df.copy() # start with a full copy of the dataframe
    if post_type != "All": # filter by post type if one is selected
        filtered_df = filtered_df[filtered_df["Post type"] == post_type]
    if gain_filter == "Gain": # filter to only posts that gained a follower if selected
        filtered_df = filtered_df[filtered_df["Follows"] >= 1]
    filtered_df = filtered_df.sort_values("Publish time") # sort chronologically for the time series
    fig = go.Figure()
    for metric in metrics: # loop through each selected metric and add a trace
        fig.add_trace(go.Scatter(x=filtered_df["Publish time"], y=filtered_df[metric], mode="lines+markers", name=metric)) # raw metric values over time
        fig.add_trace(go.Scatter(x=filtered_df["Publish time"], y=filtered_df[metric].rolling(7).mean(), mode="lines", name=f"{metric} 7-day Avg", line=dict(dash="dash"))) # 7-day rolling average for smoothing
    fig.update_layout(title="Time Series of Social Media Metrics", xaxis_title="Date", yaxis_title="Count", xaxis=dict(tickformat="%b %d, %Y")) # format the layout and axis labels
    return fig

In [ ]:
# CSV Download callback - triggers when the download button is clicked
@app.callback(
    Output("download-text-csv", "data"), # sends file data to the download component
    Input("download-text-csv-btn", "n_clicks"), # listens for button click
    State("views-filter-input", "value"), # reads current min views filter without triggering the callback
    State("likes-filter-input", "value"), # reads current min likes filter without triggering the callback
    prevent_initial_call=True # don't run this callback on page load
)
def download_text_file(n_clicks, min_views, min_likes):
    filtered_df = df.copy() # start with a full copy of the dataframe
    if min_views is not None: # apply min views filter if set
        filtered_df = filtered_df[filtered_df["Views"] >= min_views]
    if min_likes is not None: # apply min likes filter if set
        filtered_df = filtered_df[filtered_df["Likes"] >= min_likes]
    cols = ["Description"] + [c for c in filtered_df.columns if c != "Description"] # move Description to the first column
    filtered_df = filtered_df[cols] # reorder the columns
    return dcc.send_data_frame(filtered_df.to_csv, "text_analysis_filtered.csv", index=False) # send the filtered dataframe as a downloadable CSV

In [ ]:
# Heatmap Select All Days callback - triggers when "Select All Days" button is clicked
@app.callback(
    Output("heatmap_day_filter", "value"), # updates the day checklist to have all days selected
    Input("heatmap_select_all_days", "n_clicks"), # listens for button click
    prevent_initial_call=True # don't run on page load
)
def select_all_days(n):
    return [d['value'] for d in days_options] # return a list of all day values to check all boxes

# Heatmap Graph callback - triggers when metric, post type, or day filter changes
@app.callback(
    Output("heatmap_graph", "figure"), # updates the heatmap graph
    Input("heatmap_metric", "value"), # listens for metric selection
    Input("heatmap_posttype", "value"), # listens for post type filter
    Input("heatmap_day_filter", "value") # listens for day of week filter
)
def update_heatmap(metric, post_type, selected_days):
    dff = df.copy() # start with a full copy of the dataframe
    if post_type != "All": # filter by post type if one is selected
        dff = dff[dff["Post type"] == post_type]
    dff["post_time"] = dff["Publish time"].dt.hour # extract the hour of day from the publish timestamp
    dff["post_day"] = dff["Publish time"].dt.day_name() # extract the day of week from the publish timestamp
    if selected_days: # if specific days are selected, filter to only those
        dff = dff[dff["post_day"].isin(selected_days)]
    pivot = dff.groupby(["post_day", "post_time"])[metric].mean().reset_index() # average the metric by day and hour
    fig = px.density_heatmap(pivot, x="post_time", y="post_day", z=metric, color_continuous_scale=["blue","green","yellow","red"], nbinsx=24, title=f"Best Time to Post for {metric}") # build heatmap with 24 hour bins
    fig.update_layout(xaxis_title="Hour of Day (0-23)", yaxis_title="Day of Week", coloraxis_colorbar=dict(title=f"Avg {metric}")) # format axis labels and colorbar
    return fig


In [ ]:
# Posting Gap Analysis callback - triggers when metric or post type filter changes
@app.callback(
    Output("posting-gap-graph", "figure"), # updates the posting gap line graph
    Input("posting-gap-metric", "value"), # listens for metric selection
    Input("posting-gap-posttype-filter", "value") # listens for post type filter
)
def update_posting_gap_graph(metric, post_type):
    dff = df.copy() # start with a full copy of the dataframe
    if post_type != "All": # filter by post type if one is selected
        dff = dff[dff["Post type"] == post_type]
    dff = dff.sort_values("Publish time") # sort chronologically before calculating gaps
    dff['days_between_posts'] = dff['Publish time'].diff().dt.days.fillna(0) # recalculate days between posts after filtering
    summary = dff.groupby('days_between_posts', as_index=False).agg({
        'total_posts_per_row': 'sum', 'Views': 'mean', 'Likes': 'mean', 'view_like_ratio': 'mean'
    }) # aggregate metrics by posting gap
    return px.line(summary, x='days_between_posts', y=metric, markers=True, title=f"{metric} vs Days After Previous Post", labels={'days_between_posts': 'Days After Previous Post', metric: metric}) # line chart of metric vs posting gap


In [ ]:
# Post Count by Gap callback - triggers when post type filter changes
@app.callback(
    Output("days-between-post-count-graph", "figure"), # updates the post count bar chart
    Input("posting-gap-posttype-filter", "value") # listens for post type filter (shared with above callback)
)
def update_days_between_post_count(post_type):
    dff = df.copy() # start with a full copy of the dataframe
    if post_type != "All": # filter by post type if one is selected
        dff = dff[dff["Post type"] == post_type]
    dff = dff.sort_values("Publish time") # sort chronologically before calculating gaps
    dff['days_between_posts'] = dff['Publish time'].diff().dt.days.fillna(0) # recalculate days between posts after filtering
    counts = dff['days_between_posts'].value_counts().reset_index() # count how many posts fall into each gap value
    counts.columns = ['Days After Previous Post', 'Number of Posts'] # rename columns for readability
    counts = counts.sort_values('Days After Previous Post') # sort by gap size for the bar chart
    return px.bar(counts, x='Days After Previous Post', y='Number of Posts', text='Number of Posts', title="Number of Posts by Days After Previous Post") # bar chart showing post frequency by gap size

In [ ]:
# run app
if __name__ == "__main__":
    app.run(debug=True, host="127.0.0.1", port=8052) # run the app locally on port 8052 with debug mode enabled